In [1]:
import duckdb

In [2]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [7]:
df = con.execute("""
        Select * from (
            select *, 
            ROW_NUMBER() OVER(PARTITION BY NATBR ORDER BY data_ingestao DESC) as row_number  
            from bronze_produtos
            where data_ingestao >= '2025-11-01'
        ) 
        where row_number = 1""").fetch_df()
df.head(10)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao,row_number
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-06-03 20:03:45.921036,1
1,10002,MARTELO,BT50,100,500,z0019_1.csv,2026-06-03 20:03:45.921036,1
2,10004,SERRA,BT10,100,200,z0019_2.csv,2026-06-03 20:14:02.253142,1
3,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-06-03 20:14:02.253142,1
4,10003,PREGO,BT10,100,60,z0019_2.csv,2026-06-03 20:14:02.253142,1


In [11]:
df_final = df.drop(columns=['nome_arquivo', 'data_ingestao', 'row_number'])
df_final = df_final.rename(columns={"NATBR":"id"})
df_final = df_final.rename(columns={"MAKTX":"nm_produto"})
df_final = df_final.rename(columns={"WERKS":"id_categoria"})
df_final = df_final.rename(columns={"MAINS":"id_fornecedor"})
df_final = df_final.rename(columns={"LABST":"vl_preco"})
	    			
df_final.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100
1,10002,MARTELO,BT50,100,500
2,10004,SERRA,BT10,100,200
3,10005,MACHADO,BT50,100,100
4,10003,PREGO,BT10,100,60


In [14]:
df_final.dtypes


id               str
nm_produto       str
id_categoria     str
id_fornecedor    str
vl_preco         str
dtype: object

In [17]:
df2 = df_final
df2 = df2.astype({
    'id': int,
    'nm_produto': str,       
    'id_categoria': str,
    'id_fornecedor': int,
    'vl_preco': float

})
df2.dtypes

id                 int64
nm_produto           str
id_categoria         str
id_fornecedor      int64
vl_preco         float64
dtype: object

In [18]:
con.execute("""
    CREATE TABLE IF NOT EXISTS produtos (
        id BIGINT,
        nm_produto TEXT,
        id_categoria TEXT,
        id_fornecedor BIGINT,
        vl_preco FLOAT        
    )
""")

In [19]:
con.execute("insert into produtos select * from df2")

In [20]:
df_resultado = con.execute("select * from produtos").fetch_df()

df_resultado.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT50,100,500.0
2,10004,SERRA,BT10,100,200.0
3,10005,MACHADO,BT50,100,100.0
4,10003,PREGO,BT10,100,60.0


In [21]:
con.close()